<a href="https://colab.research.google.com/github/werowe/HypatiaAcademy/blob/master/ml/golf_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is **supervised learning**.  That means labeled data.

For example, if we have a picture of a cat and we want to teach the neural network to recognize cats then we features:

pixels.  parts of the cats<-----features (X)
cat or not cat<-----label


### What is Supervised Learning?

Supervised learning is a type of machine learning where the model learns from labeled data. This means that for each input example, there's a corresponding output label. The goal of the model is to learn a mapping function from the input features (X) to the output label (y).

In our case, the 'features' are whether it's 'Raining', 'Windy', or 'Cold', and the 'label' is whether or not to 'Golf'. The model learns the relationship between these weather conditions and the decision to golf.

### Why Random Forests?

Random Forests are an ensemble learning method for classification and regression. They operate by constructing a multitude of decision trees at training time and outputting the class that is the mode of the classes (classification) or mean prediction (regression) of the individual trees.

Key characteristics that make them powerful:
*   **Reduces Overfitting**: By combining multiple trees, they tend to reduce the risk of overfitting compared to a single decision tree.
*   **Handles High Dimensionality**: Can work well with a large number of features.
*   **Feature Importance**: Can provide insights into which features are most important for prediction.
*   **Robust to Noise**: Less sensitive to noise in the data.

# Question:  How you we using random data label golf


That's a very insightful question! You're right to point that out. For this sample data, I explicitly created a simplified logic to define 'Golf' based on 'Raining', 'Windy', and 'Cold'. This was done to simulate a scenario where there is a clear underlying relationship between the features and the target, which is essential for a classification model to learn and demonstrate its capability.

In a real-world application, 'Golf' would be an observed outcome – for example, based on historical records of when someone actually golfed, along with the weather conditions at that time. You wouldn't derive the 'Golf' label from the weather features; instead, you would collect independent observations of both. The purpose of this synthetic data is to create a dataset where the patterns are known, allowing us to easily test if the Random Forest model can learn those patterns.

So, while 'Golf' was generated using those features for this specific sample dataset, for the machine learning model, 'Golf' is still treated as the target label that we want to predict, and 'Raining', 'Windy', and 'Cold' are the independent features used for that prediction. Does that make sense?

In [3]:
import pandas as pd
import numpy as np

# Set a random seed for reproducibility
np.random.seed(42)

# Generate sample data
data_size = 100

raining = np.random.choice([0, 1], size=data_size)  # 0 for No, 1 for Yes
windy = np.random.choice([0, 1], size=data_size)    # 0 for No, 1 for Yes
cold = np.random.choice([0, 1], size=data_size)       # 0 for No, 1 for Yes

# Create a simplified logic for 'should_golf'
# For example: If not raining AND not too windy AND not too cold, then golf.
# Otherwise, don't golf.
should_golf = ((raining == 0) & (windy == 0) & (cold == 0)) | \
              ((raining == 0) & (windy == 0) & (cold == 1) & (np.random.rand(data_size) > 0.3)) | \
              ((raining == 0) & (windy == 1) & (cold == 0) & (np.random.rand(data_size) > 0.6))

should_golf = should_golf.astype(int) # Convert boolean to 0 or 1

# Create a DataFrame
df = pd.DataFrame({
    'Raining': raining,
    'Windy': windy,
    'Cold': cold,
    'Golf': should_golf
})

print(df.head())
print(f"\nValue counts for 'Golf':\n{df['Golf'].value_counts()}")

   Raining  Windy  Cold  Golf
0        0      0     0     1
1        1      1     1     0
2        0      1     0     1
3        0      1     0     0
4        0      1     1     0

Value counts for 'Golf':
Golf
0    75
1    25
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


### Explanation of Random Forest Principles: Bagging and Feature Sampling

#### Bagging (Bootstrap Aggregating)

Bagging is an ensemble meta-algorithm that works by training multiple models (typically of the same type) on different subsets of the training data, and then combining their predictions. For Random Forests, this means:

1.  **Bootstrapping**: Instead of training each decision tree on the entire training dataset, a Random Forest creates multiple subsets of the training data by **sampling with replacement**. This means some data points may appear multiple times in a subset, while others may not appear at all. Each of these bootstrap samples is roughly the same size as the original training dataset.
2.  **Parallel Training**: Each decision tree in the forest is trained independently on one of these bootstrap samples.
3.  **Aggregation**: For classification tasks (like our 'Golf' prediction), the final prediction is made by taking a **majority vote** of the predictions from all individual trees. If 51 out of 100 trees predict 'Golf', then the Random Forest predicts 'Golf'.

This process helps to reduce variance and overfitting, as each tree sees a slightly different view of the data, and their combined decisions are more robust than any single tree's decision.

#### Feature Sampling (Random Subspace Method)

In addition to bagging the data, Random Forests also introduce randomness at the feature selection level when building each individual decision tree:

1.  **Random Subset of Features**: When a decision tree is being built, and it needs to decide on the best split at each node, it doesn't consider all available features. Instead, it randomly selects a subset of features to choose from. For classification, the default number of features to consider at each split is the square root of the total number of features (e.g., if there are 9 features, 3 will be considered at each split).
2.  **Reduced Correlation**: This random feature selection further decorrelates the individual trees in the forest. If all trees were allowed to consider all features, they might all pick the same strong predictors early on, leading to highly correlated trees and less reduction in variance.

By combining both bagging (sampling data) and feature sampling (sampling features), Random Forests create a diverse ensemble of trees that are less prone to overfitting and often achieve higher predictive accuracy.

In [5]:
# Define features (X) and target (y)
X = df[['Raining', 'Windy', 'Cold']]
y = df['Golf']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")


X_train shape: (70, 3)
X_test shape: (30, 3)
y_train shape: (70,)
y_test shape: (30,)


In [6]:
# Initialize the Random Forest Classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

print("RandomForestClassifier initialized:")
print(rf_classifier)


RandomForestClassifier initialized:
RandomForestClassifier(random_state=42)


In [9]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the Random Forest Classifier
# n_estimators: This parameter specifies the number of trees in the forest.
# Each of these 100 trees will be trained on a bootstrap sample of the data (bagging).
# A higher number generally leads to better performance but increases computation time.
rf_classifier = RandomForestClassifier(
    n_estimators=100,  # Each tree is trained on a bootstrap sample of the data.
    random_state=42    # Ensures reproducibility of the random operations (bootstrapping, feature sampling).
) # By default, `RandomForestClassifier` uses `bootstrap=True` for bagging
  # and `max_features='sqrt'` for feature sampling (considering sqrt(n_features) at each split).

# To see details of the default parameters, you can inspect the object or its documentation.
# You can also set a 'verbose' parameter for some parts of scikit-learn, but for RandomForestClassifier training,
# detailed output of individual tree construction or bagging specifics is not directly available via a simple 'verbose' flag.
# Usually, verbose modes are for showing progress during lengthy computations, not internal algorithm mechanics.

print("RandomForestClassifier initialized:")
print(rf_classifier)

RandomForestClassifier initialized:
RandomForestClassifier(random_state=42)


In [8]:
# Train the Random Forest Classifier
rf_classifier.fit(X_train, y_train)

print("RandomForestClassifier trained successfully.")

RandomForestClassifier trained successfully.


### Training the Random Forest Model (`.fit()` method)

When `rf_classifier.fit(X_train, y_train)` is called, the following steps, incorporating bagging and feature sampling, occur:

1.  **Create `n_estimators` Trees**: The algorithm initializes to build 100 decision trees as specified by `n_estimators=100`.
2.  **Bootstrap Samples**: For each of the 100 trees, a unique bootstrap sample of the `X_train` and `y_train` data is created. This sample is roughly the same size as `X_train` but contains duplicate rows and omits others from the original `X_train`.
3.  **Build Individual Trees**: For each bootstrap sample, a decision tree is constructed. At each node of this tree:
    *   A random subset of features (by default, `sqrt(number_of_features)`) is considered for the best split. For our data with 3 features ('Raining', 'Windy', 'Cold'), this typically means considering `sqrt(3) ≈ 1 or 2` features at each split.
    *   The best split among this random subset of features is chosen to partition the data.
    *   This process continues until a stopping criterion is met (e.g., maximum depth, minimum samples per leaf).
4.  **Ensemble Creation**: Once all 100 trees are built, they collectively form the Random Forest model. Each tree is now trained and ready to make predictions.

At this stage, the model has learned the complex, non-linear relationships between the weather conditions and the 'Golf' decision by leveraging the diversity of the individual trees and their training processes.